# 04. End-to-End Pipeline

This notebook shows a small end-to-end workflow with one clear guardrail checkpoint. The point is simple: clean text flows through, uncertain text stops for review.

## Why this notebook exists
The first three notebooks isolate the pieces. This one shows how a guardrail fits inside a small pipeline, so you can see the path from raw text to structured output and where review happens when the text is uncertain.

## Concepts
- Agents as tools
- Handoffs
- Guardrail policy and enforcement
- Human review after trigger
- Tracing

## Dataset
Any `.txt` files stored in `data/` are loaded directly in the notebook.

## Colab data setup
If the notebook is opened in Google Colab without cloning the repo, upload `data.zip` from the repository root to the current working directory using the Files panel, then rerun the setup cell below.

## Further reading
- Agents: https://openai.github.io/openai-agents-python/agents
- Tools: https://openai.github.io/openai-agents-python/tools/
- Handoffs: https://openai.github.io/openai-agents-python/handoffs/
- Guardrails: https://openai.github.io/openai-agents-python/guardrails/
- Tracing: https://openai.github.io/openai-agents-python/tracing/


In [ ]:
import os
import sys
from dataclasses import dataclass
from pathlib import Path

from dotenv import load_dotenv
from agents import Agent, Runner, InputGuardrail, GuardrailFunctionOutput, handoff, trace, set_tracing_export_api_key

DEFAULT_MODEL = 'gpt-5.4-mini'

In [ ]:
candidate_dirs = [Path.cwd() / 'data', Path.cwd().parent / 'data']
data_dir = next((path for path in candidate_dirs if path.exists()), None)
if data_dir is None:
    zip_candidates = [Path.cwd() / 'data.zip', Path.cwd().parent / 'data.zip']
    zip_path = next((path for path in zip_candidates if path.exists()), None)
    if zip_path is not None:
        import zipfile
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(zip_path.parent)
        data_dir = next((path for path in candidate_dirs if path.exists()), None)
if data_dir is None:
    if 'google.colab' in sys.modules:
        print('Colab detected, but data/ is still missing.')
        print('Upload data.zip from the repository root into the Files panel, then rerun this cell.')
    else:
        raise FileNotFoundError('data/ folder not found. Clone the repo locally or place data.zip next to the notebook.')

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))

load_dotenv(Path.cwd() / '.env')
load_dotenv(Path.cwd().parent / '.env')
assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env or your environment before running this notebook.'
set_tracing_export_api_key(os.environ['OPENAI_API_KEY'])


def load_documents() -> list[dict[str, str]]:
    documents = []
    for index, path in enumerate(sorted(data_dir.glob('*.txt')), start=1):
        documents.append({
            'id': index,
            'title': path.stem.replace('_', ' ').strip().title(),
            'filename': path.name,
            'text': path.read_text(encoding='utf-8').strip(),
        })
    if not documents:
        raise FileNotFoundError('No .txt files found in data/. Add one or more documents and rerun the notebook.')
    return documents

DOCUMENTS = load_documents()
records = [{'id': item['id'], 'title': item['title'], 'text': item['text']} for item in DOCUMENTS]
records

## Step 1: Single-agent extraction

Start with the simplest part of the pipeline: one agent extracting structured details from a clean record.

In [ ]:
archive_agent = Agent(
    name='Archive Extractor',
    instructions=(
        'Read the historical text carefully. Extract people, places, dates, and evidence. '
        'Return concise JSON-like bullet points.'
    ),
    model=DEFAULT_MODEL,
)

clean_text = records[0]['text']
with trace('End-to-end single-agent extraction'):
    extraction_result = await Runner.run(archive_agent, clean_text)
print(extraction_result.final_output)

## Step 2: Build the multi-agent pipeline

Reuse the notebook 2 pattern: specialists as tools plus a coordinator handoff. This is the pipeline we will protect with guardrails.

In [ ]:
ocr_cleaner = Agent(
    name='OCR Cleaner',
    instructions='Fix obvious OCR mistakes while preserving original meaning. Do not invent new information.',
    model=DEFAULT_MODEL,
)

metadata_specialist = Agent(
    name='Metadata Specialist',
    instructions='Extract people, places, dates, and key facts from cleaned historical text.',
    model=DEFAULT_MODEL,
)

summarizer = Agent(
    name='Summarization Agent',
    instructions='Write a two-sentence summary grounded in the provided cleaned text and extracted facts.',
    model=DEFAULT_MODEL,
)

cleaner_tool = ocr_cleaner.as_tool('clean_ocr', 'Clean OCR text while preserving meaning.')
extractor_tool = metadata_specialist.as_tool('extract_metadata', 'Extract structured metadata from cleaned text.')
summarizer_tool = summarizer.as_tool('summarize_history', 'Write a short summary based on cleaned text and facts.')

handoff_metadata_specialist = Agent(
    name='Metadata Specialist',
    instructions=(
        'You receive cleaned OCR text from the coordinator. Extract people, places, dates, and key facts. '
        'Return concise evidence-based metadata.'
    ),
    model=DEFAULT_MODEL,
)

handoff_coordinator = Agent(
    name='Coordinator',
    instructions=(
        'Clean obvious OCR errors first. Then hand off to the metadata specialist. '
        'Make conservative edits and keep evidence visible.'
    ),
    tools=[cleaner_tool, extractor_tool, summarizer_tool],
    handoffs=[handoff(handoff_metadata_specialist, tool_name_override='transfer_to_metadata_specialist')],
    model=DEFAULT_MODEL,
)

handoff_metadata_specialist, handoff_coordinator

## Step 3: Add guardrail policy and enforcement

This mirrors notebook 3: a policy function decides review, and the SDK guardrail enforces that policy at runtime.

In [ ]:
@dataclass
class ReviewDecision:
    record_id: int
    uncertain: bool
    confidence: float
    notes: str

def should_review(decision: ReviewDecision) -> bool:
    return decision.uncertain or decision.confidence < 0.8

def review_decision_from_text(input_text: str) -> ReviewDecision:
    if not isinstance(input_text, str):
        return ReviewDecision(record_id=0, uncertain=True, confidence=0.0, notes='input is not text')

    text = input_text.lower()
    uncertainty_terms = ['may be', 'unclear', 'uncertain', 'blurred', 'partially legible']

    uncertainty_hits = sum(term in text for term in uncertainty_terms)

    suspicious_chars = '^~`_|/[]{}<>'
    suspicious_char_hits = sum(ch in suspicious_chars for ch in input_text)
    digit_letter_switch_hits = sum(
        (input_text[i].isalpha() and input_text[i + 1].isdigit())
        or (input_text[i].isdigit() and input_text[i + 1].isalpha())
        for i in range(len(input_text) - 1)
    )
    replacement_char_hits = input_text.count('\ufffd')

    ocr_hits = 0
    if suspicious_char_hits >= 2:
        ocr_hits += 1
    if digit_letter_switch_hits >= 2:
        ocr_hits += 1
    if replacement_char_hits >= 1:
        ocr_hits += 1

    penalty = 0.2 * uncertainty_hits + 0.1 * ocr_hits
    confidence = max(0.0, min(1.0, 1.0 - penalty))
    uncertain = uncertainty_hits > 0 or ocr_hits >= 2

    notes = []
    if uncertainty_hits:
        notes.append('explicit uncertainty language detected')
    if ocr_hits:
        notes.append('generic ocr-noise signals detected')
    if not notes:
        notes.append('clear enough to process')

    return ReviewDecision(
        record_id=0,
        uncertain=uncertain,
        confidence=round(confidence, 2),
        notes='; '.join(notes),
    )

def uncertainty_guardrail(ctx, agent, input_text):
    decision = review_decision_from_text(input_text)
    if should_review(decision):
        return GuardrailFunctionOutput(output_info=decision.notes, tripwire_triggered=True)
    return GuardrailFunctionOutput(output_info='ok', tripwire_triggered=False)

guardrail = InputGuardrail(uncertainty_guardrail, name='uncertainty_guardrail')
review_agent = Agent(
    name='Review Agent',
    instructions=(
        'Assess whether the text can be safely processed. '
        'If it is uncertain, return a short note asking for human review.'
    ),
    input_guardrails=[guardrail],
    model=DEFAULT_MODEL,
)

(guardrail, review_agent)

## Step 4: Run one process for clean and uncertain records

Both records follow the same path. If the guardrail passes, run the notebook 2 pipeline. If it triggers, show an interactive human-review widget before stopping.

> **Tip — handling rate limits:** For more resilient execution, replace `await Runner.run(...)` with the `run_with_retry()` helper introduced at the end of notebook 01. It adds automatic exponential backoff for 429 and 5xx errors with no changes to the rest of your code.

In [ ]:
from agents import InputGuardrailTripwireTriggered


def human_review_panel(record: dict, decision: ReviewDecision) -> str:
    """Render a review panel and read the human's decision from stdin.

    Why stdin and not ipywidgets: comm-message delivery is unreliable in some VS Code
    Jupyter builds. The stdin channel is part of the Jupyter wire protocol and works
    the same in VS Code, JupyterLab, Colab, and terminal IPython.
    """
    print()
    print('=' * 72)
    print(f"  HUMAN REVIEW — Record {record['id']}: {record.get('title', '')}")
    print('=' * 72)
    print(f"  Confidence: {decision.confidence}")
    print(f"  Notes:      {decision.notes}")
    print('-' * 72)
    print(record['text'])
    print('=' * 72)
    while True:
        choice = input("Decide  [a]pprove / [s]end to archive / [q]uit:  ").strip().lower()
        if choice in {'a', 'approve', 'approved'}:
            return 'approved'
        if choice in {'s', 'archive', 'archived'}:
            return 'archived'
        if choice in {'q', 'quit'}:
            raise KeyboardInterrupt('Human review cancelled')
        print("  Invalid choice — type 'a', 's', or 'q'.")


async def run_case(label: str, record: dict):
    text = record['text']
    decision = review_decision_from_text(text)
    print(f"\n--- {label} (record {record['id']}) ---")
    print('Policy decision:', decision)
    try:
        with trace(f'End-to-end guardrail {label}'):
            review_result = await Runner.run(review_agent, text)
        print('Guardrail output:', review_result.final_output)

        with trace(f'End-to-end pipeline {label}'):
            pipeline_result = await Runner.run(handoff_coordinator, text)
        print('Pipeline output:')
        print(pipeline_result.final_output)
        return
    except InputGuardrailTripwireTriggered:
        print('Guardrail triggered: InputGuardrailTripwireTriggered')

    action = human_review_panel(record, decision)
    print(f"Human decision: {action}")

    if action == 'archived':
        print('Record sent to archive for manual handling. Pipeline stopped.')
        return

    with trace(f"End-to-end pipeline human-approved record {record['id']}"):
        pipeline_result = await Runner.run(handoff_coordinator, text)
    print('Pipeline output after human approval:')
    print(pipeline_result.final_output)


await run_case('clean case', records[0])

In [ ]:
await run_case('uncertain case', records[-1])